# Semester 1 Week 4 LAB / PRACTICAL SHEET
**Canny Edge Detection Using Python OpenCV**

## PART A – Environment Setup
### Task 1: Install Required Libraries
Ensure that Python and OpenCV are installed.

(Already done)

In [ ]:
# pip install opencv-python
# pip install numpy
# pip install matplotlib

Import the required libraries:

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## PART B – Read an Image
### Task 2: Load and Display an Image

In [2]:
import cv2
image = cv2.imread("Images/images7.jpg")
cv2.imshow("Original Image", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

**Questions**
1. What does `cv2.imread()` do?
    > It loads/reads an image file from the specified file path into a NumPy matrix array.
2. What colour format does OpenCV use by default?
    > OpenCV uses BGR (Blue, Green, Red) format instead of RGB.
3. What happens if the filename is incorrect?
    > `cv2.imread()` returns None without raising an explicit file-not-found error.

## PART C – Convert to Grayscale
### Task 3: Grayscale Conversion

In [3]:
gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)
cv2.imshow("Grayscale Image", gray)
cv2.waitKey(0)
cv2.destroyAllWindows()

**Questions**
1. Why is grayscale conversion useful before edge detection?   
    > Edges depend on intensity contrast rather than color variation; reducing color channels simplifies processing and eliminates unnecessary color noise.
2. What is the pixel intensity range of an 8-bit grayscale image?
    > 0 (pure black) to 255 (pure white).
3. How does grayscale processing reduce computational requirements?
    > It reduces image data from 3 color channels (BGR) to 1 single intensity channel, cutting memory footprint and operations by 66%.  

## PART D – Gaussian Smoothing
### Task 4: Reduce Image Noise
Before detecting edges, apply a Gaussian filter.

In [4]:
blurred = cv2.GaussianBlur(
    gray,
    (5, 5),
    0
)
cv2.imshow("Gaussian Blurred Image", blurred)
cv2.waitKey(0)
cv2.destroyAllWindows()

**Purpose**

Noise can generate unwanted intensity changes that may be incorrectly detected as edges.
Gaussian smoothing reduces such noise before gradient calculation.

**Questions**
1. Why is noise reduction important before edge detection?
    > High-frequency pixel noise produces sharp local intensity spikes that derivative operators misinterpret as false edges.
2. What does (5,5) represent?
    > It defines a $5 \times 5$ pixel spatial kernel matrix used for blurring.
3. What could happen if excessive smoothing is applied?
    > Genuine boundary edges become blurred, feature localization degrades, and fine structural details disappear.

## PART E – Apply Canny Edge Detection
### **Task 5: Basic Canny Implementation**
Use OpenCV's cv2.Canny() function.

In [5]:
edges = cv2.Canny(
    blurred,
    100, # lower threshold
    200  # upper threshold
)
cv2.imshow("Canny Edge Detection", edges)
cv2.waitKey(0)
cv2.destroyAllWindows()

## PART F – Understanding the Thresholds
### Task 6: Experiment with Threshold Values
Run the Canny detector using different threshold combinations.

In [6]:
# Experiment 1
edges1 = cv2.Canny(
blurred,
50,
100
)
# Experiment 2
edges2 = cv2.Canny(
blurred,
100,
200
)
# Experiment 3
edges3 = cv2.Canny(
blurred,
150,
250
)
# Display the results:
cv2.imshow("Threshold 50-100", edges1)
cv2.imshow("Threshold 100-200", edges2)
cv2.imshow("Threshold 150-250", edges3)
cv2.waitKey(0)
cv2.destroyAllWindows()

Complete the table:

| Lower Threshold | Upper Threshold | Edge Detail | Noise | Observation |
|---|---|---|---|---|
| 50 | 100 | High | High | Captures fine details and weak edges, but retains unwanted background noise. |
| 100 | 200 | Moderate | Low | Provides a balanced, clean edge map with suppressed background noise. |
| 150 | 250 | Low | Very Low | Discards noise completely, but misses weaker genuine edges and leaves fragmented lines. |

**Questions**
1. What happens when the thresholds are too low?
    > Noise and minor intensity changes trigger false edges, producing a cluttered edge map.
2. What happens when the thresholds are too high?
    > Significant object contours are lost, leading to fragmented or missing boundaries.
3. Which threshold combination produced the best result?
    > 100 - 200 typically yields the best balance between edge preservation and noise suppression.
4. Why might different images require different thresholds?
    > Variations in lighting, contrast levels, and inherent image noise require tailored cutoffs.

## PART G – Understanding the Internal Stages
### Stage 1 – Gaussian Filtering
The image is smoothed to remove noise.

In [7]:
blurred = cv2.GaussianBlur(
    gray,
    (5, 5),
    0
)

### Stage 2 – Gradient Calculation
Canny uses image gradients to locate intensity changes.
The horizontal gradient is approximately obtained using Sobel X:
$G_x$
The vertical gradient is obtained using Sobel Y:
$G_y$

The gradient magnitude is:
$
G = \sqrt{G_x^2 + G_y^2}
$
The gradient direction is:
$
\theta =
\tan^{-1}
\left(
\frac{G_y}{G_x}
\right)
$

### Task 7: Calculate Sobel Gradients

In [8]:
sobel_x = cv2.Sobel(
    blurred,
    cv2.CV_64F,
    1,
    0,
    ksize=3
)
sobel_y = cv2.Sobel(
    blurred,
    cv2.CV_64F,
    0,
    1,
    ksize=3
)
# Calculate gradient magnitude:
magnitude = np.sqrt(
    sobel_x ** 2 + sobel_y ** 2
)
magnitude = cv2.normalize(
    magnitude,
    None,
    0,
    255,
    cv2.NORM_MINMAX
)
magnitude = magnitude.astype(np.uint8)
# Display:
cv2.imshow("Gradient Magnitude", magnitude)
cv2.waitKey(0)
cv2.destroyAllWindows()

**Questions**
1. What does the gradient magnitude represent?
    > The strength or rate of intensity change at a specific pixel.
2. What does a high gradient value indicate?
    > A sharp contrast boundary, signifying a prominent physical edge.
3. Why are Sobel gradients useful in Canny Edge Detection?
    > They compute orthogonal intensity derivatives ($G_x$ and $G_y$) needed to calculate both edge magnitude and direction.

## PART H – Non-Maximum Suppression
**Concept**

Gradient calculation may initially produce thick edges.
Non-Maximum Suppression is used to thin the edges.

For each pixel:
- The gradient direction is considered.
- The pixel is compared with neighbouring pixels along the gradient direction.
- If it is not the local maximum, it is suppressed.

The result is a much thinner edge representation.

**Question**

Explain why Non-Maximum Suppression is important in the Canny algorithm.

> Gradient operators leave broad, thick edge bands; Non-Maximum Suppression thins these responses down to 1-pixel wide lines by retaining only local intensity maxima along the gradient direction.

## PART I – Double Thresholding
After Non-Maximum Suppression, edge pixels are classified into three categories.

1. **Strong Edge**

If
$
G > T_H
$,
the pixel is classified as a strong edge.

2. **Weak Edge**

If
$
T_L \leq G \leq T_H
$,
the pixel is classified as a weak edge.

3. **Non-edge**

If
$
G < T_L
$,
the pixel is discarded.

where:
- $T_L$ = lower threshold
- $T_H$ = upper threshold

## PART J – Edge Tracking by Hysteresis
Weak edges are examined to determine whether they are connected to strong edges.
- Weak edges connected to strong edges are retained.
- Weak edges not connected to strong edges are removed.
This process is called edge tracking by hysteresis.

**Questions**
1. Why are two thresholds used instead of one?
    > Single thresholding creates broken edges from fluctuating intensities; dual thresholds isolate definite edges while giving uncertain edge pixels a second check.
2. Why are some weak edges retained?
    > Weak edges are kept if they physically connect to strong edges, indicating they are genuine extensions of a continuous boundary.
3. What happens to isolated weak edges?
    > They are discarded as background noise.

## PART K – Complete Canny Edge Detection Program
### Task 8: Write and execute the following program.

In [ ]:
import cv2
import numpy as np
# Read image
filename = input("Enter image filename: ")
image = cv2.imread(filename)
# Check whether image exists
if image is None:
    print("Error: Image could not be opened.")
    exit()
# Convert to grayscale
gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)
# Apply Gaussian smoothing
blurred = cv2.GaussianBlur(
    gray,
    (5, 5),
    0
)
# Apply Canny Edge Detection
edges = cv2.Canny(
    blurred,
    100, # lower threshold
    200 # upper threshold
)
# Display results
cv2.imshow("Original Image", image)
cv2.imshow("Grayscale Image", gray)
cv2.imshow("Gaussian Blurred Image", blurred)
cv2.imshow("Canny Edges", edges)
# Save output
cv2.imwrite(
    "canny_edges.jpg",
    edges
)
print("Edge image saved as canny_edges.jpg")
cv2.waitKey(0)
cv2.destroyAllWindows()

The **complete Python OpenCV program** that satisfies all the requirements of the practical exercise.

Try the following images:
- images.jpg
- image (14).jpg
- image (12).jpg
- image (13).jpg
- images5.jpg

In [ ]:
# ==========================================================
# Canny Edge Detection using Python OpenCV
# ==========================================================
import cv2
import os
# ----------------------------------------------------------
# Step 1: Accept an image filename
# ----------------------------------------------------------
filename = input("Enter the image filename (e.g. image.jpg): ")
# ----------------------------------------------------------
# Step 2: Read the image
# ----------------------------------------------------------
image = cv2.imread("Images/" + filename)
# Check if image exists
if image is None:
    print("Error: Unable to open the image.")
    print("Please check the filename and try again.")
    exit()
# ----------------------------------------------------------
# Step 3: Convert the image to grayscale
# ----------------------------------------------------------
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
# ----------------------------------------------------------
# Step 4: Apply Gaussian smoothing
# ----------------------------------------------------------
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
# ----------------------------------------------------------
# Step 5: Allow user to enter lower threshold
# ----------------------------------------------------------
lower_threshold = int(input("Enter the LOWER threshold (e.g. 50): "))
# ----------------------------------------------------------
# Step 6: Allow user to enter upper threshold
# ----------------------------------------------------------
upper_threshold = int(input("Enter the UPPER threshold (e.g. 150): "))
# Validate thresholds
if lower_threshold >= upper_threshold:
    print("\nWarning: Lower threshold should be smaller than the upper threshold.")
    print("Using default values: 100 and 200.\n")
    lower_threshold = 100
    upper_threshold = 200
# ----------------------------------------------------------
# Step 7: Apply Canny Edge Detection
# ----------------------------------------------------------
edges = cv2.Canny(
    blurred,
    lower_threshold,
    upper_threshold
)
# ----------------------------------------------------------
# Step 8: Display all processing stages
# ----------------------------------------------------------
cv2.imshow("1. Original Image", image)
cv2.imshow("2. Grayscale Image", gray)
cv2.imshow("3. Gaussian Blurred Image", blurred)
cv2.imshow("4. Canny Edge Detection", edges)
# ----------------------------------------------------------
# Step 9: Save the final edge image
# ----------------------------------------------------------
base_name = os.path.splitext(os.path.basename(filename))[0]
output_filename = "Output/" + base_name + "_canny_edges.jpg"
cv2.imwrite(output_filename, edges)
print("\nProcessing completed successfully.")
print("Lower Threshold :", lower_threshold)
print("Upper Threshold :", upper_threshold)
print("Output image saved as:", output_filename)
# ----------------------------------------------------------
# Wait for key press before closing windows
# ----------------------------------------------------------
cv2.waitKey(0)
cv2.destroyAllWindows()

An **interactive Canny Edge Detector** using OpenCV trackbars.
Here is a complete interactive Canny Edge Detector using OpenCV trackbars.
You can move the lower and upper threshold sliders and see the edge map update
immediately.

Try the following images:
- images.jpg
- image (14).jpg
- image (12).jpg
- image (13).jpg
- images5.jpg

In [13]:
import os
import cv2
# ----------------------------------------------------------
# 1. Read the image
# ----------------------------------------------------------
filename = input("Enter image filename (e.g. image.jpg): ")
image = cv2.imread("Images/" + filename)
if image is None:
    print("Error: Unable to open the image.")
    exit()
# ----------------------------------------------------------
# 2. Convert image to grayscale
# ----------------------------------------------------------
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
# ----------------------------------------------------------
# 3. Apply Gaussian smoothing
# ----------------------------------------------------------
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
# ----------------------------------------------------------
# 4. Create window
# ----------------------------------------------------------
window_name = "Interactive Canny Edge Detector"
cv2.namedWindow(window_name)
# ----------------------------------------------------------
# 5. Dummy callback function
# Trackbars require a callback function.
# ----------------------------------------------------------
def nothing(x):
    pass
# ----------------------------------------------------------
# 6. Create trackbars
# ----------------------------------------------------------
cv2.createTrackbar(
    "Lower Threshold",
    window_name,
    50, # Initial value
    255, # Maximum value
    nothing
)
cv2.createTrackbar(
    "Upper Threshold",
    window_name,
    150, # Initial value
    255, # Maximum value
    nothing
)
# ----------------------------------------------------------
# 7. Interactive processing loop
# ----------------------------------------------------------
while True:
    # Get current trackbar positions
    lower = cv2.getTrackbarPos(
    "Lower Threshold",
    window_name
    )
    upper = cv2.getTrackbarPos(
    "Upper Threshold",
    window_name
    )
    # Apply Canny Edge Detection
    edges = cv2.Canny(
    blurred,
    lower,
    upper
)
    # Display result
    cv2.imshow(window_name, edges)
    # Press ESC to exit
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break
# ----------------------------------------------------------
# 8. Save final edge image
# ----------------------------------------------------------
base_name = os.path.splitext(os.path.basename(filename))[0]
output_filename = "Output/" + base_name + "_canny_edges.jpg"
cv2.imwrite(output_filename, edges)
print("Final Canny edge image saved as:", output_filename)
# ----------------------------------------------------------
# 9. Close all windows
# ----------------------------------------------------------
cv2.destroyAllWindows()

Enter image filename (e.g. image.jpg):  images5.jpg


Final Canny edge image saved as: Output/images5_canny_edges.jpg


**Self-study questions:**
1. What are the three main design objectives of the Canny detector?
    > Good detection, good edge localization, and minimal multiple responses per single edge.
2. Why is Gaussian smoothing performed?
    > To suppress high-frequency noise before calculating spatial derivatives.
3. What is the purpose of gradient calculation?
    > To determine the magnitude (strength) and angle (orientation) of brightness changes.
4. What is Non-Maximum Suppression?
    > An edge-thinning method that suppresses non-peak gradient values along the gradient direction.
5. Why is Non-Maximum Suppression necessary?
    > To produce sharp, 1-pixel thick edges rather than thick gradient regions.
6. What is double thresholding?
    > Categorizing pixel gradients into Strong ($G > T_H$), Weak ($T_L \le G \le T_H$), and Non-edges ($G < T_L$)
7. What is the difference between a strong edge and a weak edge?
    > Strong edges exceed the high threshold and are unconditionally accepted; weak edges fall between thresholds and require connectivity verification.
8. What is edge tracking by hysteresis?
    > Preserving weak edges only if they form a continuous path connected to a strong edge.
9. Why are weak edges connected to strong edges retained?
    > Physical object contours naturally produce varying local contrast; connectivity proves the weak pixel is part of a real boundary.
10. How do the lower and upper thresholds affect the result?
    > $T_H$ selects core structural edges; $T_L$ controls how aggressively weak connected segments are filled in.
11. What happens if the threshold values are too low?
    > The result becomes noisy with cluttered, spurious edges.
12. What happens if the threshold values are too high?
    > Boundary lines become broken, incomplete, or absent.
13. How does noise affect Canny Edge Detection?
    > Unfiltered noise creates high-gradient false positives that lead to unwanted edge pixels.
14. Why does the Canny algorithm generally produce thinner edges than Sobel?
    > Canny incorporates Non-Maximum Suppression to thin thick gradient bands into 1-pixel lines, whereas raw Sobel does not.
15. What are the main advantages of Canny Edge Detection?
    > Superior noise immunity, thin single-pixel edge output, high localization accuracy, and low susceptibility to false edges.
16. What are its limitations?
    > Computationally heavier than basic operators, sensitive to manual threshold selection, and can blur sharp corners under heavy Gaussian filtering.

# This is the end. Thank you for watching!